In [1]:
# Author: Arthur Prigent
# Email: arthur.prigent@univ-brest.fr

In [2]:
import matplotlib.patches as mpatches
from scipy.stats import pearsonr
import matplotlib.ticker as mticker
import cartopy.crs as ccrs
import cartopy
import matplotlib
import scipy
from mpl_toolkits.axes_grid1.inset_locator import inset_axes

import matplotlib.colors as mcolors

from datetime import datetime, timedelta
import cartopy.feature as cfeature
import numpy as np
import xarray as xr
import glob
import matplotlib.pyplot as plt
import netCDF4
import string
from scipy.spatial import cKDTree
from datetime import timedelta
import gsw

# PSAL

In [3]:
# path = "/data0/user/aprigent/ISAS/ISAS23_DMFD_m*_PSAL.nc"
# files = sorted(glob.glob(path))
# print(files)
# psal_isas_01 = xr.open_dataset(files[0])
# # Open and concatenate along a new dimension
# ds_ISAS_tmp = xr.concat(
#     [xr.open_dataset(f).PSAL.isel(t=0) for f in files],
#     dim="month"
# )
# ds_ISAS = xr.Dataset({'psal':(['month','depth','latitude','longitude'],ds_ISAS_tmp.values)},
                     
#                      coords={'month': np.arange(1,13,1),
#                              'depth': psal_isas_01.depth.values,
#                              'latitude': psal_isas_01.latitude.values,
#                              'longitude': psal_isas_01.longitude.values})
# ds_ISAS.to_netcdf('/data0/user/aprigent/ISAS/ISAS23_PSAL_clim.nc')


In [4]:
# ds_all_psal = xr.open_dataset('/data0/user/aprigent/PROCESSED/level_3/psal_common_full.nc',decode_times=False)
# ds_ISAS = xr.open_dataset('/data0/user/aprigent/ISAS/ISAS23_PSAL_clim.nc')

# # ds_masks = xr.open_dataset('/data0/user/aprigent/mask_regions.nc')

# # ds_masks


# ds_all_psal_anom = ds_all_psal.copy()


# # # Interpolate basin mask to profile positions
# # basin_at_profiles = ds_masks.interp(
# #     lon=ds_all_psal_anom.longitude % 360,
# #     lat=ds_all_psal_anom.latitude,
# #     method="nearest"
# # )

# # # Only in-situ profiles in the Amerasian basin 
# # profiles_anom_amerasian_psal = ds_all_psal_anom.where(
# #     basin_at_profiles.BARENTS == 1,
# #     drop=True)

# # # Only profiles after 01/01/2005
# # # profiles_anom_amerasian_psal = profiles_anom_amerasian_psal.where(profiles_anom_amerasian_psal.time >= 731947, drop=True)
# # profiles_anom_amerasian_psal = profiles_anom_amerasian_psal

# # convert ordinal time to datetime
# times = ds_all_psal_anom.time.values
# dt_list = [datetime.fromordinal(int(t)) for t in times]


# # del ds_all_psal,ds_all_psal_anom


# profile_psal = np.array(ds_all_psal_anom.salinity_QC.values)
# lons_psal = xr.DataArray(ds_all_psal_anom.longitude.values, dims="profile")
# lats_psal = xr.DataArray(ds_all_psal_anom.latitude.values, dims="profile")

# months = xr.DataArray([dt.month for dt in dt_list], dims="profile")
    
    
# # # bi-linearly interpolate ISAS at profiles' locations    
# psal_isas_profiles = ds_ISAS.psal.interp(
#     longitude=lons_psal,
#     latitude=lats_psal)
# del ds_ISAS

# # # Take ISAS profiles at corresponding month
# psal_isas_profiles = psal_isas_profiles.sel(month=months)


# psal_anom = profile_psal - psal_isas_profiles.values


# ds_all_psal_anom['salinity_clim'] = (('profile','depth'),psal_isas_profiles.values)
# del psal_isas_profiles
# ds_all_psal_anom['salinity_QC_anom'] = (('profile','depth'),psal_anom)
# ds_all_psal_anom['source'] = ds_all_psal_anom['source'].astype(str)
# # ds_all_psal_anom.to_netcdf('/data0/user/aprigent/PROCESSED/level_4/psal_anomalies_relative_ISAS_arctic.nc')

In [5]:
import xarray as xr
import numpy as np
from datetime import datetime
import gc


def compute_psal_anomalies_per_basin(
    psal_file,
    isas_file,
    mask_file,
    basin_name,
    output_file,
    start_date=None,
):
    """
    Compute salinity anomalies relative to ISAS climatology for one basin.

    Parameters
    ----------
    psal_file : str
        Path to profile dataset.
    isas_file : str
        Path to ISAS climatology dataset.
    mask_file : str
        Path to basin mask dataset.
    basin_name : str
        Name of basin variable in mask file (e.g. 'BARENTS').
    output_file : str
        Output NetCDF filename.
    start_date : int or None
        Optional ordinal date threshold (e.g. 731947 for 2005-01-01).
    """

    print(f"\nProcessing basin: {basin_name}")

    ##########################
    # Load datasets
    ##########################
    ds_all_psal = xr.open_dataset(psal_file, decode_times=False)
    ds_ISAS = xr.open_dataset(isas_file)
    ds_masks = xr.open_dataset(mask_file)

    ##########################
    # Interpolate basin mask
    ##########################
    basin_at_profiles = ds_masks.interp(
        lon=ds_all_psal.longitude % 360,
        lat=ds_all_psal.latitude,
        method="nearest"
    )

    ##########################
    # Select basin profiles
    ##########################
    profiles = ds_all_psal.where(
        basin_at_profiles[basin_name] == 1,
        drop=True
    )

    ##########################
    # Optional time filtering
    ##########################
    if start_date is not None:
        profiles = profiles.where(
            profiles.time >= start_date,
            drop=True
        )

    ##########################
    # Convert ordinal time to datetime
    ##########################
    times = profiles.time.values
    dt_list = [datetime.fromordinal(int(t)) for t in times]

    ##########################
    # Prepare coordinates
    ##########################
    profile_psal = np.array(profiles.salinity_QC.values)

    lons_psal = xr.DataArray(
        profiles.longitude.values,
        dims="profile"
    )

    lats_psal = xr.DataArray(
        profiles.latitude.values,
        dims="profile"
    )

    months = xr.DataArray(
        [dt.month for dt in dt_list],
        dims="profile"
    )

    ##########################
    # Interpolate ISAS climatology
    ##########################
    psal_isas_profiles = ds_ISAS.psal.interp(
        longitude=lons_psal,
        latitude=lats_psal
    )

    # Select matching month
    psal_isas_profiles = psal_isas_profiles.sel(month=months)

    ##########################
    # Compute anomalies
    ##########################
    psal_anom = profile_psal - psal_isas_profiles.values

    profiles["salinity_QC_anom"] = (
        ("profile", "depth"),
        psal_anom
    )
    profiles['salinity_clim'] = (
        ('profile','depth'),
        psal_isas_profiles.values)

    # Ensure string type is NetCDF compatible
    profiles["source"] = profiles["source"].astype(str)

    ##########################
    # Save output
    ##########################
    profiles.to_netcdf(output_file)

    print(f"Saved: {output_file}")





In [6]:


psal_file = "/data0/user/aprigent/PROCESSED/level_3/psal_common_full.nc"
isas_file = "/data0/user/aprigent/ISAS/ISAS23_PSAL_clim.nc"
mask_file = "/data0/user/aprigent/mask_regions.nc"

basins = [
    "BARENTS",
    "EURASIAN",
    "AMERASIAN",
    "NORDIC",
    "AMERASIAN_SHELF",
    "SIBERIAN_SHELF",
    "BAFFIN",
    "KARA"
]

for basin in basins:

    output_file = (
        f"/data0/user/aprigent/PROCESSED/level_4/"
        f"psal_anomalies_relative_ISAS_{basin.lower()}_full_after_2005.nc"
    )

    compute_psal_anomalies_per_basin(
        psal_file=psal_file,
        isas_file=isas_file,
        mask_file=mask_file,
        basin_name=basin,
        output_file=output_file,
        start_date=731947,  # example:  for 2005-01-01
    )


Processing basin: BARENTS
Saved: /data0/user/aprigent/PROCESSED/level_4/psal_anomalies_relative_ISAS_barents_full_after_2005.nc

Processing basin: EURASIAN
Saved: /data0/user/aprigent/PROCESSED/level_4/psal_anomalies_relative_ISAS_eurasian_full_after_2005.nc

Processing basin: AMERASIAN
Saved: /data0/user/aprigent/PROCESSED/level_4/psal_anomalies_relative_ISAS_amerasian_full_after_2005.nc

Processing basin: NORDIC
Saved: /data0/user/aprigent/PROCESSED/level_4/psal_anomalies_relative_ISAS_nordic_full_after_2005.nc

Processing basin: AMERASIAN_SHELF
Saved: /data0/user/aprigent/PROCESSED/level_4/psal_anomalies_relative_ISAS_amerasian_shelf_full_after_2005.nc

Processing basin: SIBERIAN_SHELF
Saved: /data0/user/aprigent/PROCESSED/level_4/psal_anomalies_relative_ISAS_siberian_shelf_full_after_2005.nc

Processing basin: BAFFIN
Saved: /data0/user/aprigent/PROCESSED/level_4/psal_anomalies_relative_ISAS_baffin_full_after_2005.nc

Processing basin: KARA
Saved: /data0/user/aprigent/PROCESSED/lev